# 📈 Stock Analysis v1 — U.S. Equity Fundamental & Technical Screener

**Compatible with:** Google Colab · Jupyter Lab · Jupyter Notebook · VS Code

### What this notebook does
| Module | Details |
|--------|--------|
| **Bullish Scoring** | 8-point score: ATH proximity, Golden Cross, RSI, MACD, volume surge, P/E, 1M momentum |
| **ATH Detection** | Flags stocks within 5% of their 252-day all-time high |
| **Golden Cross** | MA50 crossing above MA200 |
| **Fundamentals** | P/E, Fwd P/E, EPS, revenue growth, profit margin, ROE, debt/equity |
| **Charts** | Price + MA chart, RSI panel, volume bars, bull-score heatmap |
| **Export** | CSV saved to `results/` or Google Drive |

---
> ⚠️ **Not financial advice.** For research and educational purposes only.

## 1 · Install Dependencies

In [ ]:
# Installs silently — safe to re-run
%pip install -q yfinance pandas numpy ta tabulate python-dotenv colorama matplotlib seaborn plotly

## 2 · Configuration

Edit the variables below, **or** set them as Colab Secrets (`🔑` icon in the left sidebar):

| Secret name | Purpose |
|---|---|
| `SCREENER_TICKERS` | Comma-separated ticker list |
| `ALPHA_VANTAGE_API_KEY` | Alpha Vantage key (optional) |
| `POLYGON_API_KEY` | Polygon.io key (optional) |

In [ ]:
import os

# ── Try loading from Colab Secrets, fall back to defaults ─────────────────
def _colab_secret(key, default=""):
    try:
        from google.colab import userdata
        return userdata.get(key) or default
    except Exception:
        return os.getenv(key, default)

# ── Edit these defaults directly if not using Secrets ─────────────────────
TICKERS_RAW = _colab_secret(
    "SCREENER_TICKERS",
    "AAPL,MSFT,GOOGL,AMZN,NVDA,TSLA,META,BRK-B,JPM,JNJ,V,UNH,XOM,MA,HD"
)
TICKERS      = [t.strip().upper() for t in TICKERS_RAW.split(",") if t.strip()]
PERIOD       = "1y"       # yfinance period: 1d 5d 1mo 3mo 6mo 1y 2y 5y 10y ytd max
ATH_LOOKBACK = 252        # trading days to look back for ATH
RSI_PERIOD   = 14
MA_SHORT     = 50
MA_LONG      = 200
ATH_THRESH   = 0.05       # 5% below ATH counts as "near ATH"
RESULTS_DIR  = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"✔ Scanning {len(TICKERS)} tickers: {', '.join(TICKERS)}")

## 3 · (Optional) Mount Google Drive
Run this cell only if you want results saved to your Drive.

In [ ]:
SAVE_TO_DRIVE = False   # ← set True to enable Drive save

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/stock_analysis/results"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"✔ Results will be saved to Google Drive: {RESULTS_DIR}")
else:
    print("ℹ️  Drive mount skipped — results saved locally to ./results/")

## 4 · Imports & Indicator Functions

In [ ]:
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.facecolor": "#0f0f0f",
                     "figure.facecolor": "#1a1a1a", "axes.edgecolor": "#333",
                     "text.color": "white", "axes.labelcolor": "white",
                     "xtick.color": "#aaa", "ytick.color": "#aaa",
                     "grid.color": "#2a2a2a", "grid.linestyle": "--"})
print("✔ Libraries loaded")

In [ ]:
# ── Technical Indicator Functions ─────────────────────────────────────────

def compute_rsi(series, period=14):
    delta = series.diff().dropna()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return round(float(rsi.iloc[-1]), 2)

def compute_macd(series):
    ema12  = series.ewm(span=12, adjust=False).mean()
    ema26  = series.ewm(span=26, adjust=False).mean()
    macd   = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    return round(float(macd.iloc[-1]), 4), round(float(signal.iloc[-1]), 4)

def is_above_both_mas(close, short, long_):
    return bool(close.iloc[-1] > close.rolling(short).mean().iloc[-1] and
                close.iloc[-1] > close.rolling(long_).mean().iloc[-1])

def is_golden_cross(close, short, long_):
    ma_s = close.rolling(short).mean()
    ma_l = close.rolling(long_).mean()
    return bool(ma_s.iloc[-1] > ma_l.iloc[-1] and ma_s.iloc[-2] <= ma_l.iloc[-2])

def near_ath(close, lookback, threshold=0.05):
    ath = close.tail(lookback).max()
    return bool((ath - close.iloc[-1]) / ath <= threshold)

def pct_below_ath(close, lookback):
    ath = close.tail(lookback).max()
    return round(float(((ath - close.iloc[-1]) / ath) * 100), 2)

def volume_surge(volume, window=20):
    avg = volume.tail(window).mean()
    return round(float(volume.iloc[-1] / avg), 2) if avg else 0.0

def price_change_pct(close, days):
    if len(close) < days + 1: return 0.0
    return round(float((close.iloc[-1] - close.iloc[-days-1]) / close.iloc[-days-1] * 100), 2)

def bullish_score(row):
    signals = []
    if row.get("above_ma50_ma200"):                                     signals.append("Price > MA50 & MA200")
    if row.get("golden_cross"):                                         signals.append("Golden Cross")
    if row.get("near_ath"):                                             signals.append("Near ATH (≤5%)")
    if row.get("rsi") and 50 < row["rsi"] < 70:                        signals.append(f"RSI bullish ({row['rsi']})")
    if row.get("macd_line") and row.get("macd_signal") \
       and row["macd_line"] > row["macd_signal"]:                       signals.append("MACD > Signal")
    if row.get("volume_ratio") and row["volume_ratio"] > 1.5:           signals.append(f"Vol surge x{row['volume_ratio']}")
    if row.get("pe_ratio") and 0 < row["pe_ratio"] < 30:               signals.append(f"P/E attractive ({row['pe_ratio']})")
    if row.get("ret_1m") and row["ret_1m"] > 3:                         signals.append(f"1M +{row['ret_1m']}%")
    return len(signals), signals

print("✔ Indicator functions defined")

## 5 · Fetch & Analyse All Tickers

In [ ]:
def analyze_ticker(ticker):
    try:
        stock = yf.Ticker(ticker)
        hist  = stock.history(period=PERIOD)
        if hist.empty or len(hist) < MA_LONG + 5:
            print(f"  ⚠️  {ticker}: insufficient history")
            return None
        info   = stock.info or {}
        close  = hist["Close"]
        volume = hist["Volume"]
        macd_line, macd_signal = compute_macd(close)
        row = {
            "ticker":        ticker,
            "price":         round(float(close.iloc[-1]), 2),
            "ret_1d":        price_change_pct(close, 1),
            "ret_1m":        price_change_pct(close, 21),
            "ret_3m":        price_change_pct(close, 63),
            "rsi":           compute_rsi(close, RSI_PERIOD),
            "macd_line":     macd_line,
            "macd_signal":   macd_signal,
            "ma50":          round(float(close.rolling(MA_SHORT).mean().iloc[-1]), 2),
            "ma200":         round(float(close.rolling(MA_LONG).mean().iloc[-1]), 2),
            "above_ma50_ma200": is_above_both_mas(close, MA_SHORT, MA_LONG),
            "golden_cross":  is_golden_cross(close, MA_SHORT, MA_LONG),
            "near_ath":      near_ath(close, ATH_LOOKBACK, ATH_THRESH),
            "pct_below_ath": pct_below_ath(close, ATH_LOOKBACK),
            "volume_ratio":  volume_surge(volume),
            "market_cap":    info.get("marketCap"),
            "pe_ratio":      info.get("trailingPE"),
            "fwd_pe":        info.get("forwardPE"),
            "eps_ttm":       info.get("trailingEps"),
            "revenue_growth":info.get("revenueGrowth"),
            "earnings_growth":info.get("earningsGrowth"),
            "profit_margin": info.get("profitMargins"),
            "roe":           info.get("returnOnEquity"),
            "debt_equity":   info.get("debtToEquity"),
            "sector":        info.get("sector", "N/A"),
            "industry":      info.get("industry", "N/A"),
            "_hist":         hist,   # kept for charting, dropped before CSV
        }
        score, signals = bullish_score(row)
        row["bull_score"]   = score
        row["bull_signals"] = "; ".join(signals) if signals else "—"
        return row
    except Exception as exc:
        print(f"  ✗ {ticker}: {exc}")
        return None

# ── Run screener ───────────────────────────────────────────────────────────
print(f"Fetching data for {len(TICKERS)} tickers…\n")
results = []
for i, ticker in enumerate(TICKERS, 1):
    print(f"  [{i:>2}/{len(TICKERS)}] {ticker}", end="  ", flush=True)
    row = analyze_ticker(ticker)
    if row:
        results.append(row)
        print(f"score={row['bull_score']}/8  rsi={row['rsi']}  {'✔ near ATH' if row['near_ath'] else ''}")

print(f"\n✅ Done — {len(results)}/{len(TICKERS)} tickers retrieved.")

## 6 · Results Table

In [ ]:
def fmt_cap(v):
    if not v: return "N/A"
    if v >= 1e12: return f"${v/1e12:.2f}T"
    if v >= 1e9:  return f"${v/1e9:.2f}B"
    return f"${v/1e6:.2f}M"

def build_display_df(results):
    rows = []
    for r in sorted(results, key=lambda x: -x["bull_score"]):
        rows.append({
            "Ticker":     r["ticker"],
            "Price":      f"${r['price']:,.2f}",
            "1D %":       f"{r['ret_1d']:+.2f}%",
            "1M %":       f"{r['ret_1m']:+.2f}%",
            "3M %":       f"{r['ret_3m']:+.2f}%",
            "RSI":        r["rsi"],
            "P/E":        f"{r['pe_ratio']:.1f}" if r["pe_ratio"] else "N/A",
            "Mkt Cap":    fmt_cap(r["market_cap"]),
            "Below ATH":  f"{r['pct_below_ath']}%",
            "Vol Ratio":  r["volume_ratio"],
            "Near ATH":   "✅" if r["near_ath"]     else "❌",
            "GoldenX":    "✅" if r["golden_cross"]  else "❌",
            "Score /8":   r["bull_score"],
            "Signals":    r["bull_signals"],
        })
    return pd.DataFrame(rows)

df_display = build_display_df(results)

def color_score(val):
    if isinstance(val, int):
        if val >= 5: return "background-color: #1a4d1a; color: #7fff7f"
        if val >= 3: return "background-color: #4d3d00; color: #ffd966"
        return "background-color: #4d0000; color: #ff9999"
    return ""

styled = (df_display.style
    .applymap(color_score, subset=["Score /8"])
    .set_properties(**{"font-size": "12px"})
    .set_table_styles([{"selector": "th", "props": [("background-color", "#2d2d2d"),
                                                     ("color", "white"), ("font-size", "12px")]}]))
display(styled)

## 7 · Interactive Price Chart (Plotly)
Select any ticker from the dropdown to view price, MA50/200, volume, and RSI.

In [ ]:
def plot_ticker(row):
    hist   = row["_hist"]
    close  = hist["Close"]
    volume = hist["Volume"]
    ma50   = close.rolling(MA_SHORT).mean()
    ma200  = close.rolling(MA_LONG).mean()

    # RSI series
    delta    = close.diff().dropna()
    gain     = delta.clip(lower=0).ewm(alpha=1/RSI_PERIOD, adjust=False).mean()
    loss     = (-delta.clip(upper=0)).ewm(alpha=1/RSI_PERIOD, adjust=False).mean()
    rsi_ser  = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=[f"{row['ticker']} — Price & Moving Averages", "Volume", "RSI(14)"],
        vertical_spacing=0.05
    )

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=hist.index, open=hist["Open"], high=hist["High"],
        low=hist["Low"], close=close, name="Price",
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350"
    ), row=1, col=1)
    fig.add_trace(go.Scatter(x=hist.index, y=ma50,  name=f"MA{MA_SHORT}",  line=dict(color="#ffd700", width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=hist.index, y=ma200, name=f"MA{MA_LONG}", line=dict(color="#ff6b35", width=1.5)), row=1, col=1)

    # Volume bars
    colors = ["#26a69a" if c >= o else "#ef5350"
              for c, o in zip(hist["Close"], hist["Open"])]
    fig.add_trace(go.Bar(x=hist.index, y=volume, name="Volume",
                         marker_color=colors, opacity=0.7), row=2, col=1)

    # RSI
    fig.add_trace(go.Scatter(x=rsi_ser.index, y=rsi_ser, name="RSI",
                             line=dict(color="#ab47bc", width=1.5)), row=3, col=1)
    fig.add_hline(y=70, line_dash="dot", line_color="red",   opacity=0.5, row=3, col=1)
    fig.add_hline(y=30, line_dash="dot", line_color="green", opacity=0.5, row=3, col=1)

    fig.update_layout(
        template="plotly_dark", height=700,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02),
        margin=dict(t=60, b=20)
    )
    fig.show()

# Plot the highest-scoring ticker by default
top = max(results, key=lambda x: x["bull_score"])
print(f"Showing chart for top-scoring ticker: {top['ticker']} (score {top['bull_score']}/8)")
plot_ticker(top)

In [ ]:
# ── Plot any ticker from your list ────────────────────────────────────────
CHART_TICKER = "NVDA"   # ← change this

match = next((r for r in results if r["ticker"] == CHART_TICKER.upper()), None)
if match:
    plot_ticker(match)
else:
    print(f"'{CHART_TICKER}' not found in results. Available: {[r['ticker'] for r in results]}")

## 8 · Bullish Score Heatmap

In [ ]:
sorted_results = sorted(results, key=lambda x: -x["bull_score"])
tickers_sorted = [r["ticker"]    for r in sorted_results]
scores         = [r["bull_score"] for r in sorted_results]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Bar chart: bull score ──────────────────────────────────────────────────
bar_colors = ["#26a69a" if s >= 4 else ("#ffd700" if s >= 2 else "#ef5350") for s in scores]
axes[0].barh(tickers_sorted[::-1], scores[::-1], color=bar_colors[::-1], edgecolor="none")
axes[0].set_xlabel("Bullish Score (out of 8)", color="white")
axes[0].set_title("Bullish Score by Ticker", color="white", fontsize=13)
axes[0].axvline(4, color="white", linestyle="--", alpha=0.4, label="Strong threshold")
axes[0].legend(facecolor="#2d2d2d", labelcolor="white")

# ── Scatter: RSI vs 1-Month Return ────────────────────────────────────────
rsis  = [r["rsi"]    for r in sorted_results]
ret1m = [r["ret_1m"] for r in sorted_results]
sc = axes[1].scatter(rsis, ret1m, c=scores, cmap="RdYlGn", s=120, edgecolors="white", linewidths=0.5, vmin=0, vmax=8)
for r in sorted_results:
    axes[1].annotate(r["ticker"], (r["rsi"], r["ret_1m"]),
                     textcoords="offset points", xytext=(6, 4), fontsize=8, color="#ccc")
axes[1].axvline(70, color="red",   linestyle="--", alpha=0.4, label="RSI overbought")
axes[1].axvline(30, color="green", linestyle="--", alpha=0.4, label="RSI oversold")
axes[1].axhline(0,  color="white", linestyle="-",  alpha=0.2)
axes[1].set_xlabel("RSI (14)",           color="white")
axes[1].set_ylabel("1-Month Return (%)", color="white")
axes[1].set_title("RSI vs 1M Return  (colour = bull score)", color="white", fontsize=13)
axes[1].legend(facecolor="#2d2d2d", labelcolor="white")
plt.colorbar(sc, ax=axes[1], label="Bull Score")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "overview_chart.png"), bbox_inches="tight", facecolor="#1a1a1a")
plt.show()
print("✔ Chart saved to results/overview_chart.png")

## 9 · Fundamentals Deep-Dive

In [ ]:
def pct(v):
    return f"{v*100:.1f}%" if v is not None else "N/A"

fund_rows = []
for r in sorted(results, key=lambda x: -x["bull_score"]):
    fund_rows.append({
        "Ticker":          r["ticker"],
        "Sector":          r["sector"],
        "P/E (TTM)":       f"{r['pe_ratio']:.1f}"   if r["pe_ratio"]       else "N/A",
        "Fwd P/E":         f"{r['fwd_pe']:.1f}"     if r["fwd_pe"]         else "N/A",
        "EPS (TTM)":       f"${r['eps_ttm']:.2f}"   if r["eps_ttm"]        else "N/A",
        "Rev Growth":      pct(r["revenue_growth"]),
        "Earn Growth":     pct(r["earnings_growth"]),
        "Profit Margin":   pct(r["profit_margin"]),
        "ROE":             pct(r["roe"]),
        "Debt/Equity":     f"{r['debt_equity']:.1f}" if r["debt_equity"]   else "N/A",
        "Bull Score":      r["bull_score"],
    })

df_fund = pd.DataFrame(fund_rows)

def highlight_pe(val):
    try:
        v = float(str(val).replace("N/A","99"))
        if 0 < v < 20: return "color: #7fff7f"
        if v < 35:     return "color: #ffd966"
        return "color: #ff9999"
    except: return ""

display(df_fund.style
    .applymap(highlight_pe, subset=["P/E (TTM)", "Fwd P/E"])
    .applymap(color_score,  subset=["Bull Score"])
    .set_properties(**{"font-size": "12px"})
    .set_table_styles([{"selector": "th", "props": [("background-color", "#2d2d2d"),
                                                     ("color", "white"), ("font-size", "12px")]}]))

## 10 · Export Results to CSV

In [ ]:
from datetime import datetime

ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(RESULTS_DIR, f"analysis_{ts}.csv")

# Drop internal history object before saving
export_rows = [{k: v for k, v in r.items() if k != "_hist"} for r in results]
df_export   = pd.DataFrame(export_rows)
df_export   = df_export.sort_values("bull_score", ascending=False)
df_export.to_csv(csv_path, index=False)

print(f"✔ CSV saved → {csv_path}")
print(f"   {len(df_export)} rows  ·  {len(df_export.columns)} columns")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(csv_path)
    print("✔ Download triggered in Colab")
except ImportError:
    print("ℹ️  Not in Colab — file saved locally.")

display(df_export.head(5))

## 11 · Final Summary

In [ ]:
bullish     = [r for r in results if r["bull_score"] >= 4]
near_ath_l  = [r for r in results if r["near_ath"]]
golden_l    = [r for r in results if r["golden_cross"]]

print("═" * 55)
print(f"  📊 Stock Analysis v1 — Summary ({datetime.now().strftime('%Y-%m-%d')})")
print("═" * 55)
print(f"  Tickers scanned  : {len(results)}")
print(f"  Strong bullish   : {len(bullish)}  (score ≥ 4/8)")
print(f"  Near ATH (≤5%)   : {len(near_ath_l)}  — {[r['ticker'] for r in near_ath_l]}")
print(f"  Golden Cross     : {len(golden_l)}  — {[r['ticker'] for r in golden_l]}")
print()
print("  🏆 Top 5 by Bullish Score:")
for r in sorted(results, key=lambda x: -x["bull_score"])[:5]:
    bar = "█" * r["bull_score"] + "░" * (8 - r["bull_score"])
    print(f"     {r['ticker']:<6}  [{bar}]  {r['bull_score']}/8  —  {r['bull_signals']}")
print("═" * 55)